## Section 1 -- Background and Hyperparameter Rationale

This notebook tunes the hyperparameters of a **Temporal Convolutional Network (TCN)**
for binary EEG seizure detection (ictal vs non-ictal) using **Optuna** with the
Tree-structured Parzen Estimator (TPE) sampler.

### Receptive field formula

Following Bai et al. (2018), each residual block contains **two** dilated
causal convolutions at the same dilation factor. For a TCN with `L` blocks,
kernel size `k`, and exponential dilation schedule `d_l = 2^l`, the
receptive field (RF) is:

```
RF = 2 * (2^L - 1) * (k - 1) + 1
```

The factor of 2 accounts for the two convolutions per block, each
contributing `(k-1) * d` to the receptive field at dilation `d`.

At 500 Hz, 1 second = 500 samples. We enforce RF >= 500 as a hard constraint.

**Worked examples:**

| L (blocks) | k (kernel) | RF (samples) | RF (seconds) | Passes check? |
|------------|------------|--------------|--------------|---------------|
| 5          | 3          | 125          | 0.25         | No            |
| 6          | 7          | 757          | 1.51         | Yes           |
| 7          | 5          | 1017         | 2.03         | Yes           |
| 8          | 3          | 1021         | 2.04         | Yes           |
| 9          | 7          | 6133         | 12.27        | Yes           |

### Hyperparameters to tune

| Hyperparameter | Role                                      | Type         | Range / Choices           |
|----------------|-------------------------------------------|--------------|---------------------------|
| num_layers     | Number of two-conv residual blocks (Bai et al.); expands RF exponentially | int          | 5 -- 9                    |
| kernel_size    | Expands RF linearly; odd values only       | categorical  | 3, 5, 7                   |
| num_filters    | Representational width per layer           | categorical  | 32, 64, 128               |
| dropout        | Spatial dropout rate per conv block        | float        | 0.10 -- 0.50 (step 0.05)  |
| learning_rate  | AdamW step size                            | float (log)  | 1e-4 -- 1e-2              |
| weight_decay   | L2 regularisation on weights               | float (log)  | 1e-5 -- 1e-3              |
| batch_size     | Segments per gradient update               | categorical  | 16, 32, 64                |

**Justification:**

- **num_layers 5--9:** with the two-convolution-per-block design (Bai et al.,
  2018), L=5 with k=3 gives RF=125 (below threshold), but L=5 with k>=5
  passes. Beyond 9 blocks the RF exceeds the segment length.
- **kernel_size 3/5/7:** odd values only -- even kernels cause asymmetric
  causal padding and introduce subtle temporal boundary artefacts.
- **num_filters 32/64/128:** powers of two for efficient GPU memory alignment;
  values above 128 risk overfitting given the limited number of mice.
- **dropout 0.10--0.50:** lower bound prevents excessive regularisation that
  causes underfitting; upper bound prevents training collapse.
- **learning_rate 1e-4--1e-2 on log scale:** log scale is essential because
  the optimal lr can differ by orders of magnitude across configurations.
- **weight_decay 1e-5--1e-3 on log scale:** complements dropout; searched
  on log scale for the same reason as learning_rate.
- **batch_size 16/32/64:** values below 16 produce noisy gradient estimates;
  values above 64 are impractical at 2500 samples per segment on most
  research GPUs; powers of two maximise GPU memory throughput.

### Fixed architectural choices (not tuned)

**Layer normalisation (not batch normalisation):**
EEG amplitude varies across mice even after preprocessing. Batch normalisation
computes statistics across the batch dimension, making it statistically unreliable
at small batch sizes (16--64) and sensitive to between-subject amplitude differences.
Layer normalisation computes statistics per sample across the channel dimension,
so it is unaffected by batch composition.

**Spatial dropout (nn.Dropout1d):**
Standard dropout randomly zeroes individual scalar values. In a 1-D convolutional
network, adjacent time steps within a feature map carry correlated information.
Spatial dropout drops entire feature-map channels at once, which is a stronger
form of regularisation better suited to structured EEG representations.

**Weighted cross-entropy loss (BCEWithLogitsLoss with pos_weight):**
Ictal segments are rare relative to non-ictal. pos_weight = n_non_ictal / n_ictal
scales the loss on positive (ictal) samples so the model does not simply predict
the majority class. This is computed from training counts only.

**Class imbalance correction (offline downsampling + pos_weight=1.0):**
Class imbalance is addressed by offline stratified downsampling of the non-ictal
class to a 1:4 ictal:non-ictal ratio via `filter_unpaired_subjects()` and
`downsample_non_ictal()` from `tcn_utils.py`. Subjects with no ictal
representation (m254) are excluded prior to downsampling. `pos_weight` is set
to 1.0 in `BCEWithLogitsLoss` because the 1:4 downsampling is the sole
imbalance correction mechanism. This strategy is consistent across all pipeline
scripts and tuning notebooks.

**Two convolutions per residual block (Bai et al., 2018):**
Each residual block contains two dilated causal convolutions at the same
dilation factor. This provides two non-linear transformations per dilation
level before advancing to the next scale, yielding richer feature
extraction without increasing the number of dilation levels.

**Exponential dilation schedule (d_l = 2^l):**
The standard TCN default that maximises receptive field growth per additional block.
Each block doubles the dilation, so the RF grows exponentially with depth.

**AdamW with cosine annealing:**
AdamW decouples weight decay from the gradient update, preventing the regularisation
effect from shrinking as the learning rate decays. Cosine annealing gradually
reduces the learning rate from its initial value to near zero following a cosine
curve, avoiding abrupt drops that can destabilise training.

**GELU activation:**
A smooth approximation to ReLU that allows small negative gradients to flow.
Empirically well-suited to bio-signal feature extraction because EEG patterns
are smooth and continuous, not sharply thresholded.

**Global average pooling:**
After the final convolutional layer, global average pooling collapses the temporal
dimension (2500 time steps) to a single value per channel. This produces a
fixed-length vector regardless of minor variations in effective sequence length
after causal trimming, and acts as a strong spatial regulariser.

**Residual (skip) connections with 1x1 projection:**
Each convolutional block adds its input to its output. If the input and output
channel counts differ, a 1x1 convolution projects the input to match. This
stabilises gradient flow in deep stacks and prevents the network from losing
information as depth increases.

### How Optuna TPE works

**What is a trial?**
Each trial is one complete training run with a specific set of hyperparameters.
Optuna selects the hyperparameters, trains the model, evaluates it, and records
the result.

**What does TPE do differently from random search?**
Random search picks hyperparameters uniformly at random. TPE (Tree-structured
Parzen Estimator) builds two probability models after the first 15 trials:
one for hyperparameter configurations that produced good results, and one for
configurations that produced poor results. It then samples new configurations
that are likely under the "good" model and unlikely under the "bad" model.
This concentrates the search on promising regions of the hyperparameter space.

**What does the pruner do?**
The MedianPruner monitors validation F1 at each epoch. If a trial is performing
below the median of all previous trials at the same epoch, it is stopped early.
This saves compute by abandoning clearly poor configurations without waiting
for all 100 epochs.

**Early stopping (patience = 10):**
Within each trial, if the validation macro F1 does not improve for 10 consecutive
epochs, training stops and the best weights from the best epoch are restored.
This prevents overfitting and wasted computation on trials that have converged.

### GPU acceleration in this context

Optuna's search logic (TPE sampler, pruner, trial bookkeeping) runs entirely
on **CPU**. However, the computationally expensive operations inside each trial
-- the forward pass, loss computation, backward pass, and weight update --
all execute on the **GPU** when one is available. The result is that each trial
completes significantly faster, allowing more configurations to be evaluated
within a fixed time budget. Data is transferred to the GPU batch-by-batch
(not all at once) to avoid exhausting GPU memory on large datasets.

## Section 2 -- Install Dependencies

Run this cell only if packages are not already installed.

In [ ]:
# -- Section 2: Install dependencies (skip if already installed) ---------------
# !pip install torch optuna numpy scikit-learn matplotlib seaborn  # uncomment if needed

## Section 3 -- Imports, Reproducibility, and Device Configuration

In [ ]:
# -*- coding: utf-8 -*-
# -- Section 3: Imports, reproducibility, device detection ---------------------

import json                          # save hyperparameters and summary as JSON
import csv                           # write study results to CSV
import time                          # measure trial duration
import logging                       # structured logging to file and console
from pathlib import Path             # cross-platform file path handling
from datetime import datetime        # ISO 8601 timestamp for summary

import numpy as np                   # numerical operations on arrays
import torch                         # deep learning framework

import optuna                        # hyperparameter optimisation framework
from optuna.samplers import TPESampler  # Tree-structured Parzen Estimator
from optuna.pruners import MedianPruner  # prune underperforming trials

import matplotlib                    # plotting backend configuration
matplotlib.use('Agg')               # non-interactive backend for cluster use
import matplotlib.pyplot as plt      # plotting API

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress Optuna's verbose output

# -- Import shared utilities from tcn_utils.py ---------------------------------
from tcn_utils import (
    set_seed,
    make_loader,
    filter_unpaired_subjects,
    downsample_non_ictal,
    TCN,
    count_parameters,
    evaluate,
    run_training,
)

set_seed(42)  # set global seed immediately

# -- Device detection ----------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_ver = torch.version.cuda
    print(f"Device        : {gpu_name}")
    print(f"VRAM total    : {vram_gb:.2f} GB")
    print(f"CUDA version  : {cuda_ver}")
else:
    gpu_name = "cpu"
    print("No GPU detected -- training will run on CPU.")
    print("Tuning 60 trials on CPU may take considerably longer.")

print(f"PyTorch       : {torch.__version__}")
print(f"Optuna        : {optuna.__version__}")
print(f"Using device  : {DEVICE}")

## Research Reporting Checklist

The following parameters must be reported in the methods section of the paper.
Full justifications are in `tcn_utils.py` under each `RESEARCH REPORTING NOTE` block.

### Architecture
- [ ] Number of TCN layers (L) and resulting receptive field in seconds at 500 Hz: RF = 2^L * (k-1) / 500
- [ ] Kernel size (k)
- [ ] Number of filters per layer
- [ ] Dilation schedule: exponential d_l = 2^l
- [ ] Normalisation: layer normalisation (justify over batch norm)
- [ ] Dropout type: spatial (Dropout1d) and rate
- [ ] Activation function: GELU
- [ ] Pooling: global average pooling
- [ ] Total trainable parameters

### Training
- [ ] Optimiser: AdamW with weight decay value
- [ ] Learning rate and schedule: cosine annealing with eta_min
- [ ] Batch size
- [ ] Maximum epochs and early stopping patience
- [ ] Gradient clipping max_norm
- [ ] Class imbalance handling: pos_weight only (no oversampling). Report ictal:non-ictal ratio in training set
- [ ] Loss function: BCEWithLogitsLoss with pos_weight value

### Hyperparameter Tuning
- [ ] Tuning framework: Optuna with TPE sampler
- [ ] Number of trials
- [ ] Number of startup (random) trials before TPE
- [ ] Pruner: MedianPruner with n_startup_trials and n_warmup_steps
- [ ] Tuning metric: macro F1-score on validation set
- [ ] Search ranges for all seven hyperparameters (table)

### Data
- [ ] Sampling frequency: 500 Hz
- [ ] Segment length: 5 seconds (2500 samples)
- [ ] Overlap: 2.5 seconds (1250 samples)
- [ ] Split strategy: by mouse, 70/10/20
- [ ] Number of mice per partition
- [ ] Segment counts per partition (ictal and non-ictal separately)
- [ ] Preprocessing: robust z-score normalisation (no additional normalisation at load time)

## Section 4 -- Configuration

All named constants are defined here. Paths use `pathlib.Path` throughout.
The test partition is intentionally excluded -- it is reserved for a separate
evaluation script and must not be referenced anywhere in this notebook.

In [ ]:
# -- Section 4: Configuration --------------------------------------------------

# -- Data splits ---------------------------------------------------------------
# All training and tuning scripts load data from data_splits.json, produced by
# generate_data_splits.py. This ensures consistent train/val partitions across
# the entire pipeline and inherits the mouse-level leakage check.
SPLITS_PATH = Path("/scratch/22206468/INPUT_DATA/data_splits_outputs/data_splits.json")

# -- Signal parameters ---------------------------------------------------------
FS            = 500    # sampling rate in Hz
SEGMENT_LEN   = 2500   # samples per segment: 5 s * 500 Hz

# -- Training protocol ---------------------------------------------------------
MAX_EPOCHS    = 100    # maximum epochs per trial before early stopping
ES_PATIENCE   = 10     # early stopping patience: epochs without val F1 improvement
GRAD_CLIP     = 1.0    # maximum gradient norm for gradient clipping
SEED          = 42     # random seed for reproducibility

# -- Optuna configuration ------------------------------------------------------
N_TRIALS      = 60     # total number of Optuna trials
N_STARTUP     = 15     # random exploration trials before TPE kicks in
STUDY_NAME    = "tcn_HPT_binary_optuna"  # Optuna study name

# -- Output --------------------------------------------------------------------
OUTPUT_DIR    = Path("/home/people/22206468/scratch/OUTPUT/MODEL1_OUTPUT/TCNtuning_outputs")       # directory for all saved outputs
OUTPUT_DIR.mkdir(exist_ok=True)       # create if it does not exist

# -- Logging setup -------------------------------------------------------------
log = logging.getLogger("tcn_hpt")    # named logger for this notebook
log.setLevel(logging.INFO)            # minimum log level
log.handlers.clear()                  # clear handlers from previous runs

fh = logging.FileHandler(OUTPUT_DIR / "tcn_HPT_binary.log", mode="a", encoding="utf-8")  # file handler
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))  # timestamp format

ch = logging.StreamHandler()          # console handler
ch.setFormatter(logging.Formatter("%(asctime)s | %(message)s"))  # same format

log.addHandler(fh)                    # attach file handler
log.addHandler(ch)                    # attach console handler

log.info(f"Configuration loaded. Device: {DEVICE}")
log.info(f"Output directory: {OUTPUT_DIR.resolve()}")

## Section 5 -- Dataset and DataLoader

Train and validation file-label pairs are loaded from `data_splits.json`, produced
by `generate_data_splits.py`. This is the single source of truth for all pipeline
scripts and notebooks. Using the same JSON file guarantees that:

- The exact same files are used across hyperparameter tuning (this notebook),
  attention tuning (`tune_temporal_attention.py`), multi-scale tuning
  (`tune_multiscale_tcn.py`), and all training scripts (`TCN.py`, etc.).
- The mouse-level leakage check performed by `generate_data_splits.py` is inherited
  -- no mouse appears in both train and val partitions.

Each `.npy` file is loaded on demand (not all at once) to keep memory usage low.
The segments were already normalised using robust z-score (median and MAD) during
preprocessing. No additional normalisation is applied at load time.

Class imbalance is addressed by offline stratified downsampling of the non-ictal
class to a 1:4 ratio via `filter_unpaired_subjects()` and `downsample_non_ictal()`
from `tcn_utils.py`, applied before DataLoader construction. `pos_weight=1.0` in
`BCEWithLogitsLoss` ensures the downsampling is the sole correction.
Both training and validation DataLoaders are built via `make_loader()`.
For training (`train=True`), the loader shuffles the downsampled corpus.
For validation (`train=False`), a plain sequential loader is used.

In [ ]:
# -- Section 5: Dataset and DataLoader -----------------------------------------
# Load train/val file-label pairs from data_splits.json (single source of truth).

if not SPLITS_PATH.exists():
    raise FileNotFoundError(
        f"data_splits.json not found at {SPLITS_PATH}. "
        f"Run generate_data_splits.py --no-test first.")

log.info(f"Loading splits from: {SPLITS_PATH}")

with open(SPLITS_PATH, "r", encoding="utf-8") as _f:
    _splits = json.load(_f)

# -- Convert records to (filepath, label) tuples for make_loader() -------------
train_pairs = [(rec["filepath"], rec["label"]) for rec in _splits["train"]]
val_pairs   = [(rec["filepath"], rec["label"]) for rec in _splits["val"]]

if not train_pairs:
    raise RuntimeError("Train partition is empty in data_splits.json.")
if not val_pairs:
    raise RuntimeError("Val partition is empty in data_splits.json.")

# -- Corpus preparation --------------------------------------------------------
# Step 1: remove subjects with no ictal segments.
train_pairs = filter_unpaired_subjects(train_pairs, logger=log)
# Step 2: downsample non-ictal to 1:4 ratio, stratified by recording.
train_pairs = downsample_non_ictal(train_pairs, ratio=4, seed=42)
# Step 3: pos_weight = 1.0 (downsampling is the sole imbalance correction).
POS_WEIGHT_VAL = 1.0
log.info(f"Post-downsampling corpus: {len(train_pairs)} segments")
# -- End corpus preparation ----------------------------------------------------

# -- Class statistics (post-downsampling) --------------------------------------
n_train_ictal     = sum(1 for _, l in train_pairs if l == 1)
n_train_non_ictal = sum(1 for _, l in train_pairs if l == 0)
n_val_ictal       = sum(1 for _, l in val_pairs if l == 1)
n_val_non_ictal   = sum(1 for _, l in val_pairs if l == 0)

log.info(f"Train: {n_train_ictal} ictal + {n_train_non_ictal} non-ictal "
         f"= {len(train_pairs)} total ({100*n_train_ictal/max(len(train_pairs),1):.1f}% ictal)")
log.info(f"Val:   {n_val_ictal} ictal + {n_val_non_ictal} non-ictal "
         f"= {len(val_pairs)} total ({100*n_val_ictal/max(len(val_pairs),1):.1f}% ictal)")
log.info(f"pos_weight = {POS_WEIGHT_VAL:.4f}")

## Section 6 -- TCN Architecture

The model is moved to DEVICE immediately after instantiation with `.to(DEVICE)`.
This single call transfers all submodules, parameters, and buffers (including
LayerNorm statistics) to the target device.

> **`CausalConvBlock`**, **`TCN`**, and **`count_parameters`** are defined in
> `tcn_utils.py` and imported in the cell above. Each `CausalConvBlock`
> contains **two** dilated causal convolutions at the same dilation factor,
> following Bai et al. (2018). See `tcn_utils.py` for the full
> implementation, inline documentation, and research reporting notes
> describing which parameters must be reported in the methods section.

## Section 7 -- Training and Evaluation Utilities

Input and label tensors are moved to the device **batch-by-batch** inside
the training and evaluation loops. This avoids loading the entire dataset
into GPU memory at once, which would exhaust VRAM for large datasets.

Best model weights are stored on CPU (`best_state`) to avoid occupying
a second copy of the model in GPU memory.

> **`train_one_epoch`**, **`evaluate`**, and **`run_training`** are defined in
> `tcn_utils.py` and imported in the cell above. See `tcn_utils.py` for the
> full implementation, inline documentation, and research reporting notes
> describing which parameters must be reported in the methods section.
>
> `run_training` accepts an optional `trial` parameter for Optuna integration
> and computes `pos_weight` internally from `train_pairs`.

## Section 8 -- Optuna Objective Function

Each call to `optuna_objective(trial)` is one complete experiment:
Optuna proposes hyperparameters, the model is built, trained, evaluated,
and the best validation F1 is returned. Optuna uses this value to guide
the next proposal via TPE.

In [ ]:
# -- Section 8: Optuna Objective Function --------------------------------------

def optuna_objective(trial):
    """Optuna objective: train a TCN with proposed hyperparameters, return val F1.

    Parameters
    ----------
    trial : optuna.trial.Trial

    Returns
    -------
    best_val_f1 : float
    """
    # -- Sample hyperparameters ------------------------------------------------
    num_layers = trial.suggest_int("num_layers", 5, 9)                 # depth of the TCN
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7])  # odd kernel sizes only
    num_filters = trial.suggest_categorical("num_filters", [32, 64, 128])  # channel width
    dropout = trial.suggest_float("dropout", 0.10, 0.50, step=0.05)   # spatial dropout rate
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)   # AdamW learning rate
    wd = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)    # L2 regularisation
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64]) # segments per batch

    # -- Check receptive field constraint --------------------------------------
    # RF = 2*(2^L - 1)*(k - 1) + 1: two convolutions per block (Bai et al., 2018)
    rf = 2 * (2 ** num_layers - 1) * (kernel_size - 1) + 1
    if rf < 500:                                 # must cover at least 1 second at 500 Hz
        raise optuna.exceptions.TrialPruned()    # reject this configuration immediately

    # -- Build model -----------------------------------------------------------
    set_seed(SEED)                               # ensure reproducible weight initialisation
    model = TCN(num_layers, num_filters, kernel_size, dropout).to(DEVICE)  # move model to GPU/CPU

    log.info(f"Trial {trial.number}: L={num_layers} k={kernel_size} f={num_filters} "
             f"drop={dropout:.2f} lr={lr:.2e} wd={wd:.2e} bs={batch_size} "
             f"RF={rf} params={count_parameters(model)}")

    # -- Build data loaders ----------------------------------------------------
    # Class imbalance handled by offline downsampling (applied once before
    # Optuna loop). make_loader(train=True) shuffles the downsampled corpus.
    train_loader = make_loader(train_pairs, batch_size, train=True, device=DEVICE)
    val_loader = make_loader(val_pairs, batch_size, train=False, device=DEVICE)

    # -- Train and evaluate (pass train_pairs for pos_weight, trial for Optuna) -
    best_val_f1 = run_training(
        model, train_loader, val_loader,
        lr=lr, weight_decay=wd,
        max_epochs=MAX_EPOCHS, patience=ES_PATIENCE,
        device=DEVICE, train_pairs=train_pairs,
        trial=trial
    )

    # -- Save checkpoint for this trial ----------------------------------------
    ckpt_path = OUTPUT_DIR / f"trial_{trial.number:03d}.pt"
    torch.save(model.cpu().state_dict(), ckpt_path)  # save on CPU for device-agnostic loading

    return best_val_f1

## Section 9 -- Run the Hyperparameter Search

The study is created with TPESampler and MedianPruner. A callback logs
each completed trial. After all trials, results are printed and GPU
memory is released.

In [ ]:
# -- Section 9: Run the Hyperparameter Search ----------------------------------

def trial_callback(study, trial):
    """Callback executed after each completed trial."""
    if trial.state == optuna.trial.TrialState.COMPLETE:
        p = trial.params
        log.info(f"  [Trial {trial.number:3d}] F1={trial.value:.4f} "
                 f"L={p['num_layers']} k={p['kernel_size']} f={p['num_filters']} "
                 f"lr={p['learning_rate']:.2e} device={DEVICE.type}")


# -- Create study --------------------------------------------------------------
# SQLite storage enables resume after crash: re-running the notebook picks up
# from the last completed trial. load_if_exists=True loads the existing study
# if the database already contains one with the same study_name.
# To start fresh, delete the .db file: rm OUTPUT_DIR/tcn_hpt.db
sampler = TPESampler(seed=SEED, n_startup_trials=N_STARTUP)  # TPE with 15 random starts
pruner  = MedianPruner(n_startup_trials=N_STARTUP, n_warmup_steps=15)  # prune below median

STUDY_DB = OUTPUT_DIR / "tcn_hpt.db"
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",        # maximise validation macro F1
    sampler=sampler,
    pruner=pruner,
    storage="sqlite:///" + str(STUDY_DB),
    load_if_exists=True,
)

log.info(f"Optuna storage: {STUDY_DB} | completed trials so far: {len(study.trials)}")
log.info(f"Starting Optuna study: {N_TRIALS} trials, TPE sampler, MedianPruner")
log.info(f"Training device: {DEVICE.type}")

study.optimize(
    optuna_objective,
    n_trials=N_TRIALS,
    callbacks=[trial_callback],
    show_progress_bar=False      # disabled for cluster/log compatibility
)

# -- Print results -------------------------------------------------------------
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]

log.info(f"\n{'='*60}")
log.info(f"Study complete: {len(completed)} completed, {len(pruned)} pruned "
         f"out of {len(study.trials)} total")

best = study.best_trial
bp = best.params
# RF = 2*(2^L - 1)*(k - 1) + 1: two convolutions per block (Bai et al., 2018)
best_rf = 2 * (2 ** bp["num_layers"] - 1) * (bp["kernel_size"] - 1) + 1
rf_check = "PASS" if best_rf >= 500 else "WARNING: RF < 500"

log.info(f"Best trial     : {best.number}")
log.info(f"Best val F1    : {best.value:.6f}")
log.info(f"  num_layers   : {bp['num_layers']}")
log.info(f"  kernel_size  : {bp['kernel_size']}")
log.info(f"  num_filters  : {bp['num_filters']}")
log.info(f"  dropout      : {bp['dropout']:.2f}")
log.info(f"  learning_rate: {bp['learning_rate']:.2e}")
log.info(f"  weight_decay : {bp['weight_decay']:.2e}")
log.info(f"  batch_size   : {bp['batch_size']}")
log.info(f"  RF           : {best_rf} samples ({best_rf/FS:.2f} s) [{rf_check}]")
log.info(f"  Device       : {DEVICE.type}")

# -- Release GPU memory --------------------------------------------------------
if torch.cuda.is_available():
    torch.cuda.empty_cache()     # free cached GPU memory
    log.info("GPU cache cleared.")

## Section 10 -- Visualise Tuning Results

All plots are saved to the outputs/ directory as PNG files.
The matplotlib Agg backend is used so plots render without a display server.

In [ ]:
# -- Section 10: Visualise Tuning Results --------------------------------------

completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_nums = [t.number for t in completed_trials]    # trial indices
trial_f1s  = [t.value for t in completed_trials]     # corresponding val F1 values

# -- Plot 1: F1 history -------------------------------------------------------
running_best = np.maximum.accumulate(trial_f1s)      # running best F1 up to each trial
best_idx = int(np.argmax(trial_f1s))                 # index of the best trial in the list

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(trial_nums, trial_f1s, alpha=0.6, s=30, label="Trial F1")  # scatter of all F1 values
ax.plot(trial_nums, running_best, color="red", linewidth=2, label="Running best")  # best-so-far line
ax.scatter([trial_nums[best_idx]], [trial_f1s[best_idx]],
           color="gold", s=150, zorder=5, edgecolors="black", marker="*",
           label=f"Best: {trial_f1s[best_idx]:.4f}")  # highlight best trial
ax.set_xlabel("Trial number")
ax.set_ylabel("Validation macro F1")
ax.set_title("Hyperparameter Tuning -- F1 History")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "tuning_f1_history.png", dpi=150)  # save before showing
plt.close(fig)
log.info(f"Saved: {OUTPUT_DIR / 'tuning_f1_history.png'}")

# -- Plot 2: F1 distribution --------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(trial_f1s, bins=20, edgecolor="black", alpha=0.7)  # histogram of F1 values
best_f1 = max(trial_f1s)
ax.axvline(best_f1, color="red", linestyle="--", linewidth=2,
           label=f"Best: {best_f1:.4f}")  # vertical line at best F1
ax.set_xlabel("Validation macro F1")
ax.set_ylabel("Count")
ax.set_title("Hyperparameter Tuning -- F1 Distribution")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "tuning_f1_distribution.png", dpi=150)
plt.close(fig)
log.info(f"Saved: {OUTPUT_DIR / 'tuning_f1_distribution.png'}")

# -- Plot 3: Hyperparameter importance ----------------------------------------
try:
    importances = optuna.importance.get_param_importances(study)  # compute importance scores
    params_sorted = list(importances.keys())      # parameter names sorted by importance
    values_sorted = list(importances.values())    # corresponding importance values

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["tomato" if i == 0 else "steelblue" for i in range(len(params_sorted))]  # highlight top
    ax.barh(params_sorted[::-1], values_sorted[::-1], color=colors[::-1])  # horizontal bars
    ax.set_xlabel("Importance")
    ax.set_title("Hyperparameter Importance")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "hyperparameter_importance.png", dpi=150)
    plt.close(fig)
    log.info(f"Saved: {OUTPUT_DIR / 'hyperparameter_importance.png'}")
except Exception as e:
    log.warning(f"Could not compute hyperparameter importance: {e}")

# -- Plot 4: Parallel coordinate plot -----------------------------------------
try:
    from optuna.visualization.matplotlib import plot_parallel_coordinate
    fig = plot_parallel_coordinate(study)          # Optuna's built-in parallel coordinate plot
    fig.figure.tight_layout()
    fig.figure.savefig(OUTPUT_DIR / "parallel_coordinates.png", dpi=150)
    plt.close(fig.figure)
    log.info(f"Saved: {OUTPUT_DIR / 'parallel_coordinates.png'}")
except Exception as e:
    log.warning(f"Could not create parallel coordinate plot: {e}")

## Section 11 -- Save All Tuning Outputs

Three files are saved:

- **best_params.json** -- The winning hyperparameters. Used by the retraining
  script to rebuild the best model architecture and train on the full
  train+val dataset before final test evaluation.
- **study_results.csv** -- One row per completed trial. Useful for post-hoc
  analysis of which configurations worked and which did not.
- **tuning_summary.json** -- Metadata about the study: timestamps, trial
  counts, device info, and signal parameters. Provides a complete audit
  trail for reproducibility.

In [ ]:
# -- Section 11a: Save best hyperparameters ------------------------------------

bp = study.best_trial.params
# RF = 2*(2^L - 1)*(k - 1) + 1: two convolutions per block (Bai et al., 2018)
best_rf = 2 * (2 ** bp["num_layers"] - 1) * (bp["kernel_size"] - 1) + 1

best_params = {
    "best_trial_number": study.best_trial.number,
    "best_val_f1": round(study.best_trial.value, 6),
    "receptive_field_samples": best_rf,
    "receptive_field_seconds": round(best_rf / FS, 4),
    "training_device": DEVICE.type,
    "hyperparameters": {
        "num_layers":    bp["num_layers"],
        "kernel_size":   bp["kernel_size"],
        "num_filters":   bp["num_filters"],
        "dropout":       bp["dropout"],
        "learning_rate": bp["learning_rate"],
        "weight_decay":  bp["weight_decay"],
        "batch_size":    bp["batch_size"]
    }
}

params_path = OUTPUT_DIR / "best_params.json"
with open(params_path, "w") as f:
    json.dump(best_params, f, indent=2)
log.info(f"Saved: {params_path.resolve()}")

In [ ]:
# -- Section 11b: Save full study results as CSV -------------------------------

csv_path = OUTPUT_DIR / "study_results.csv"
fieldnames = ["trial_number", "val_f1", "num_layers", "kernel_size", "num_filters",
              "dropout", "learning_rate", "weight_decay", "batch_size",
              "duration_seconds", "device"]

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for t in completed:
        dur = (t.datetime_complete - t.datetime_start).total_seconds() if t.datetime_complete else 0
        writer.writerow({
            "trial_number":    t.number,
            "val_f1":          round(t.value, 6),
            "num_layers":      t.params["num_layers"],
            "kernel_size":     t.params["kernel_size"],
            "num_filters":     t.params["num_filters"],
            "dropout":         t.params["dropout"],
            "learning_rate":   t.params["learning_rate"],
            "weight_decay":    t.params["weight_decay"],
            "batch_size":      t.params["batch_size"],
            "duration_seconds": round(dur, 1),
            "device":          DEVICE.type
        })

log.info(f"Saved: {csv_path.resolve()}")

In [ ]:
# -- Section 11c: Save tuning summary -----------------------------------------

summary = {
    "timestamp":              datetime.now().isoformat(),
    "study_name":             STUDY_NAME,
    "n_trials_requested":     N_TRIALS,
    "n_trials_completed":     len([t for t in study.trials
                                   if t.state == optuna.trial.TrialState.COMPLETE]),
    "n_trials_pruned":        len([t for t in study.trials
                                   if t.state == optuna.trial.TrialState.PRUNED]),
    "best_trial_number":      study.best_trial.number,
    "best_val_f1":            round(study.best_trial.value, 6),
    "training_device":        DEVICE.type,
    "gpu_name":               gpu_name,
    "fs_hz":                  FS,
    "segment_len_samples":    SEGMENT_LEN,
    "segment_len_seconds":    SEGMENT_LEN / FS
}

summary_path = OUTPUT_DIR / "tuning_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
log.info(f"Saved: {summary_path.resolve()}")

# -- Confirm all outputs -------------------------------------------------------
log.info("")
log.info("All tuning outputs saved successfully:")
log.info(f"  1. {params_path.resolve()}")
log.info(f"  2. {csv_path.resolve()}")
log.info(f"  3. {summary_path.resolve()}")
log.info("Tuning notebook complete.")

## Parameters to Report in a Research Paper

When reporting TCN hyperparameter tuning with Optuna, the following parameters and details should be included for full reproducibility and methodological transparency.

### 1. Search Space Definition

| Hyperparameter | Search range | Sampling strategy | Description |
|----------------|-------------|-------------------|-------------|
| num_layers | [5, 9] | Uniform integer | Number of two-convolution residual blocks (Bai et al., 2018). Controls depth and receptive field. |
| kernel_size | {3, 5, 7} | Categorical | Temporal convolution kernel width (odd values only). Larger kernels capture longer local patterns per layer. |
| num_filters | {32, 64, 128} | Categorical | Output channels per convolutional layer. Controls representational capacity at each temporal scale. |
| dropout | [0.10, 0.50], step 0.05 | Discrete uniform | Spatial dropout rate (Dropout1d). Entire channels are dropped to regularise. |
| learning_rate | [1e-4, 1e-2] | Log-uniform | AdamW initial learning rate. Log-uniform ensures equal exploration across orders of magnitude. |
| weight_decay | [1e-5, 1e-3] | Log-uniform | AdamW decoupled L2 regularisation coefficient. |
| batch_size | {16, 32, 64} | Categorical | Segments per training batch. Affects gradient noise, memory, and effective learning rate. |

### 2. Derived Architectural Parameters

| Parameter | Formula / Value | Description |
|-----------|----------------|-------------|
| Receptive field (RF) | 2*(2^L - 1)*(k - 1) + 1 samples | Maximum temporal context visible to the final layer. Two convolutions per block (Bai et al., 2018). Hard constraint: RF >= 500 samples (1.0 s at 500 Hz). |
| RF in seconds | RF / fs | Receptive field in seconds for interpretability. |
| Dilation schedule | 2^i for layer i = 0, 1, ..., num_layers-1 | Exponential dilation doubles per layer for efficient long-range coverage. |
| Total trainable parameters | count_parameters(model) | Sum of all learnable weights and biases for the best trial. |

### 3. Fixed Training Protocol

| Parameter | Value | Description |
|-----------|-------|-------------|
| Maximum epochs | 100 | Upper bound on training duration per trial. |
| Early stopping patience | 10 epochs | Halt if validation macro F1 does not improve for 10 consecutive epochs. |
| Gradient clipping norm | 1.0 | Maximum L2 norm; gradients exceeding this are rescaled. |
| Optimiser | AdamW | Adam with decoupled weight decay (Loshchilov and Hutter, 2019). |
| LR schedule | Cosine annealing, T_max=max_epochs, eta_min=lr*0.01 | Smoothly decays LR from initial value to near zero. |
| Loss function | BCEWithLogitsLoss | Binary cross-entropy with built-in sigmoid. pos_weight = n_non_ictal / n_ictal. |
| Classification threshold | 0.5 | Sigmoid output >= 0.5 is predicted as ictal. |
| Random seed | 42 | Fixed seed for Python, NumPy, PyTorch (CPU+CUDA), and Optuna sampler. |
| Input normalisation | None at load time | Segments were already robust z-scored (median/MAD) during preprocessing. No additional normalisation is applied in the dataset loader. |

### 4. Fixed Architectural Choices

| Choice | Value | Description |
|--------|-------|-------------|
| Block structure | Two dilated causal Conv1d per residual block (Bai et al., 2018) | Provides two non-linear transformations per dilation level. |
| Activation | GELU | Smooth, non-monotonic activation suited to biological signals. |
| Normalisation | LayerNorm (per time step, across channels) | Preferred over BatchNorm for small-batch and variable-amplitude EEG. |
| Residual connections | 1x1 Conv projection when channels differ; identity otherwise | Stabilises gradient flow in deep stacks. |
| Pooling | Global average pooling over time | Collapses temporal dimension to fixed-length vector before the head. |
| Classification head | Linear(num_filters, 1) | Maps pooled features to a scalar logit. |
| Causal padding | Left-pad by (kernel_size-1)*dilation, trim right | Prevents access to future samples -- critical for temporal EEG. |

### 5. Optuna Search Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| Number of trials | 60 | Total optimisation budget. |
| Sampler | TPE (Tree-structured Parzen Estimator) | Bayesian method modelling good vs. bad trial distributions. |
| Startup trials | 15 | Random exploration before TPE begins informed sampling. |
| Pruner | MedianPruner | Prunes trials below median F1 at intermediate epochs. |
| Pruner warmup | 15 epochs | Pruning disabled for first 15 epochs per trial. |
| Direction | Maximise validation macro F1 | Macro F1 treats both classes equally regardless of prevalence. |
| RF constraint | RF >= 500 samples (1.0 s) | Configurations below this are immediately pruned. |

### 6. Class Imbalance Handling

| Mechanism | Where applied | Description |
|-----------|--------------|-------------|
| pos_weight in BCEWithLogitsLoss | Loss function | Set to n_non_ictal / n_ictal. Upweights ictal gradient contribution. |
| Offline downsampling | Training corpus | Non-ictal class downsampled to 1:4 ratio via filter_unpaired_subjects() and downsample_non_ictal(). pos_weight=1.0 (sole correction). |
| Macro F1 as metric | Optuna objective + early stopping | Equal weight to both classes in evaluation. |

### 7. Data and Input Specification

| Parameter | Value | Description |
|-----------|-------|-------------|
| Sampling rate | 500 Hz | Native EDF sampling rate. |
| Segment length | 2500 samples (5.0 s) | Fixed-length input window. |
| Input channels | 1 | Single-channel EEG. |
| Input shape | (batch, 1, 2500) | 1-D time series with one channel. |
| Preprocessing | Robust z-score (median/MAD) applied per chunk during preprocessing | No further normalisation at training time. |

### 8. What to Report for the Best Trial

- All seven tuned hyperparameter values
- Receptive field in samples and seconds
- Total trainable parameters
- Best validation macro F1
- Number of training epochs before early stopping
- Trial number (out of total trials attempted)
- Number of completed vs. pruned trials
- pos_weight value used
- Hardware (GPU model, VRAM) and software versions (Python, PyTorch, Optuna, CUDA)

## End of notebook

This notebook is now complete. The best hyperparameters have been saved to
`outputs/best_params.json`. The next step is to use a separate retraining
script that loads these parameters, rebuilds the TCN with the winning
configuration, trains on the combined train+val partition, and evaluates
on the held-out test set.

In [ ]:
# -- End of notebook -----------------------------------------------------------
log.info("Notebook execution finished.")